# Análisis Citus — PostgreSQL Distribuido + Búsqueda Textual
**BD2 Lab 16: MongoDB Distribuido vs Citus**

Dataset: News Headlines About President Milei (2,224 registros únicos)

**Nota:** Este notebook funciona sin conexión a Citus para tablas/gráficos.
Para `EXPLAIN ANALYZE` requiere el cluster Citus activo.

In [ ]:
import pandas as pd
import json
from datetime import datetime
import matplotlib.pyplot as plt

DATA_FILE = r"../../dataset/processed/milei_news_clean.json"

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "news_analysis_pg",
    "user": "postgres",
    "password": "postgres",
}

df = pd.read_json(DATA_FILE)
print(f"Registros cargados: {len(df)}")
df.head(3)

---
## P6: Estadísticas del Dataset y configuración Citus

In [ ]:
print("=== TABLA DE ESTADÍSTICAS ===")
print(f"Registros totales: {len(df)}")
print(f"Secciones únicas: {df['section'].nunique()}")
print(f"Periódicos únicos: {df['news_paper'].nunique()}")
print(f"Registros sin summary: {(df['summary'] == '').sum()}")
print(f"Registros sin fecha: {df['published'].isna().sum()}")

print(f"\nDistribución de secciones (top 15):")
print(df['section'].value_counts().head(15).to_string())

print(f"\nDistribución de periódicos:")
print(df['news_paper'].value_counts().to_string())

## P7: Estrategia de distribución Citus

Al igual que en MongoDB, se usa `section` como clave de distribución con **hash**.
Citus distribuye las filas entre los workers según el hash de `section`.

Ventajas:
- Q4 (GROUP BY section) es local a cada worker
- Q5 (GROUP BY news_paper) requiere scatter/gather

In [ ]:
section_dist = df['section'].value_counts()
print(f"Cardinalidad de section: {len(section_dist)}")
print(f"Estrategia elegida: HASH")
print(f"Razón: Distribución sesgada (Política domina) -> hash balancea entre workers")

---
## Q1–Q5: Queries Citus

### Requiere conexión a Citus (cluster activo)

In [ ]:
CITUS_AVAILABLE = False
try:
    import psycopg2
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    CITUS_AVAILABLE = True
    print("Conexión a Citus exitosa.")
except Exception as e:
    print(f"No se pudo conectar a Citus: {e}")
    print("Las queries requieren el cluster activo.")

In [ ]:
if CITUS_AVAILABLE:
    cursor.execute("SELECT * FROM master_get_active_worker_nodes()")
    workers = cursor.fetchall()
    print("=== Workers activos ===")
    for w in workers:
        print(f"  {w[0]}:{w[1]}")
else:
    print("Workers: pendiente (activo cluster)")

### Q1: Búsqueda textual múltiples términos

In [ ]:
if CITUS_AVAILABLE:
    import time
    
    sql = """
    SELECT title, section, 
           ts_rank(to_tsvector('spanish', coalesce(title,'') || ' ' || coalesce(summary,'')),
                   to_tsquery('spanish', 'dolar & inflacion')) AS rank
    FROM milei_news
    WHERE to_tsvector('spanish', coalesce(title,'') || ' ' || coalesce(summary,'')) @@ to_tsquery('spanish', 'dolar & inflacion')
    ORDER BY rank DESC
    LIMIT 20;
    """
    
    start = time.time()
    cursor.execute(sql)
    results = cursor.fetchall()
    elapsed = time.time() - start
    
    cursor.execute("EXPLAIN ANALYZE " + sql)
    explain = cursor.fetchall()
    
    print(f"Q1 - Tiempo: {elapsed*1000:.2f} ms")
    print(f"Q1 - Resultados: {len(results)}")
    print(f"\nTop 5:")
    for r in results[:5]:
        print(f"  [{r[2]:.3f}] {r[0][:60]} - {r[1]}")
    print(f"\n--- EXPLAIN ANALYZE ---")
    for line in explain[:20]:
        print(line[0])
else:
    print("Q1: Requiere conexión Citus.")

### Q2: Búsqueda con exclusión

In [ ]:
if CITUS_AVAILABLE:
    import time
    
    sql = """
    SELECT title, published
    FROM milei_news
    WHERE to_tsvector('spanish', coalesce(title,'') || ' ' || coalesce(summary,'')) @@ to_tsquery('spanish', 'Milei & !Twitter')
    ORDER BY published DESC
    LIMIT 20;
    """
    
    start = time.time()
    cursor.execute(sql)
    results = cursor.fetchall()
    elapsed = time.time() - start
    
    cursor.execute("EXPLAIN ANALYZE " + sql)
    explain = cursor.fetchall()
    
    print(f"Q2 - Tiempo: {elapsed*1000:.2f} ms")
    print(f"Q2 - Resultados: {len(results)}")
    print(f"\nTop 5:")
    for r in results[:5]:
        print(f"  {r[1]} - {r[0][:60]}")
    print(f"\n--- EXPLAIN ANALYZE ---")
    for e in explain[:20]:
        print(e[0])
else:
    print("Q2: Requiere conexión Citus.")

### Q3: Top 10 por fecha

In [ ]:
if CITUS_AVAILABLE:
    import time
    
    sql = """
    SELECT title, published, section, news_paper
    FROM milei_news
    ORDER BY published DESC
    LIMIT 10;
    """
    
    start = time.time()
    cursor.execute(sql)
    results = cursor.fetchall()
    elapsed = time.time() - start
    
    cursor.execute("EXPLAIN ANALYZE " + sql)
    explain = cursor.fetchall()
    
    print(f"Q3 - Tiempo: {elapsed*1000:.2f} ms")
    print(f"\nTop 10:")
    for i, r in enumerate(results, 1):
        print(f"  {i}. {r[1]} - {r[0][:60]}")
    print(f"\n--- EXPLAIN ANALYZE ---")
    for e in explain[:20]:
        print(e[0])
else:
    print("Q3: Requiere conexión Citus.")

### Q4: Agregación por shard key (section)

In [ ]:
if CITUS_AVAILABLE:
    import time
    
    sql = """
    SELECT section, COUNT(*) as cnt
    FROM milei_news
    GROUP BY section
    ORDER BY cnt DESC;
    """
    
    start = time.time()
    cursor.execute(sql)
    results = cursor.fetchall()
    elapsed = time.time() - start
    
    cursor.execute("EXPLAIN ANALYZE " + sql)
    explain = cursor.fetchall()
    
    print(f"Q4 - Tiempo: {elapsed*1000:.2f} ms")
    print(f"\nSecciones (top 10):")
    for r in results[:10]:
        print(f"  {r[0]}: {r[1]}")
    print(f"\n--- EXPLAIN ANALYZE ---")
    for e in explain[:25]:
        print(e[0])
else:
    print("Q4: Requiere conexión Citus.")

### Q5: Agregación por atributo no particionado (news_paper)

In [ ]:
if CITUS_AVAILABLE:
    import time
    
    sql = """
    SELECT news_paper, COUNT(*) as cnt
    FROM milei_news
    GROUP BY news_paper
    ORDER BY cnt DESC;
    """
    
    start = time.time()
    cursor.execute(sql)
    results = cursor.fetchall()
    elapsed = time.time() - start
    
    cursor.execute("EXPLAIN ANALYZE " + sql)
    explain = cursor.fetchall()
    
    print(f"Q5 - Tiempo: {elapsed*1000:.2f} ms")
    print(f"\nPeriódicos:")
    for r in results:
        print(f"  {r[0]}: {r[1]}")
    print(f"\n--- EXPLAIN ANALYZE ---")
    for e in explain[:25]:
        print(e[0])
else:
    print("Q5: Requiere conexión Citus.")

## P8: Análisis del índice GIN para búsqueda textual

El índice GIN (Generalized Inverted Index) en PostgreSQL:
- Almacena un **índice invertido** de lexemas a posiciones
- `to_tsvector('spanish', texto)` tokeniza, hace stemming y elimina stop words
- `ts_rank` implementa una variante de **TF-IDF** con normalización
- El índice GIN se distribuye entre workers de Citus (cada worker tiene su propio GIN local)
- `Custom Scan (Citus)` en EXPLAIN indica que Citus distribuye la consulta a los workers

## P10: Paralelización en Citus

Revisar EXPLAIN ANALYZE de Q4 vs Q5 para ver:
- Q4 (GROUP BY section): `Task Count: 2` (un task por worker), merge local rápido
- Q5 (GROUP BY news_paper): `Task Count: 2` + scatter/gather (coordinator mergea resultados)
- Presencia de `Custom Scan (Citus)` indica distribución

In [ ]:
print("=== Resumen para informe ===")
print(f"Registros totales: {len(df)}")
print(f"Secciones: {df['section'].nunique()}")
print(f"Periódicos: {df['news_paper'].nunique()}")
print(f"Distribución: HASH sobre section")